## Text Chunking

Split extracted text into smaller chunks for better retrieval and embedding.

### Import Libraries

In [1]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from pathlib import Path
import json

### Define Data Paths

In [2]:
# Define base paths - use extracted txt files from Notebook 01
BASE_DIR = Path("../data")
TXT_DIR = BASE_DIR / "txt"

### Load Extracted Text Files

In [3]:
# Load all txt files extracted from PDFs and DOCXs
try:
    documents = SimpleDirectoryReader(str(TXT_DIR)).load_data()
    print(f"Loaded {len(documents)} documents")
except FileNotFoundError:
    print(f"TXT directory not found: {TXT_DIR}")
    print("Run Notebook 01 first to extract text files")
    documents = []
except Exception as e:
    print(f"Error loading documents: {e}")
    documents = []

Loaded 4 documents


### Explore Loaded Document

In [4]:
# Print metadata of first document
if documents:
    print("First document metadata:")
    print(json.dumps(documents[0].metadata, indent=2))

First document metadata:
{
  "file_path": "/Users/apple/Documents/GitHub Repositories/Lexi-Bot/notebooks/../data/txt/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_name": "A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_type": "text/plain",
  "file_size": 20795,
  "creation_date": "2026-03-12",
  "last_modified_date": "2026-03-12"
}


In [5]:
# Print character count of each document
if documents:
    print("\nDocument sizes:")
    for doc in documents:
        file_name = doc.metadata.get('file_name', 'unknown')
        print(f"  {file_name}: {len(doc.text):,} characters")


Document sizes:
  A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt: 20,684 characters
  A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt: 21,587 characters
  A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt: 25,181 characters
  A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt: 26,467 characters


### Why Chunking?

> **Note:** Large documents need to be split into smaller chunks because:
> - Embedding models have token limits
> - Smaller chunks = more precise retrieval
> - Better context for LLM responses

### Create Sentence Splitter

In [6]:
# Create splitter with default chunk size (512 tokens)
splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print(f"Splitter created: chunk_size=512, chunk_overlap=50")

Splitter created: chunk_size=512, chunk_overlap=50


### Split Documents into Nodes

In [7]:
# Split all documents into nodes (chunks)
if documents:
    nodes = splitter.get_nodes_from_documents(documents)
    print(f"Created {len(nodes)} chunks from {len(documents)} documents")

Created 57 chunks from 4 documents


### Explore Chunks

In [8]:
# Print first chunk details
if nodes:
    print("First chunk:")
    print(f"  Text length: {len(nodes[0].text)} characters")
    print(f"  Metadata: {json.dumps(nodes[0].metadata, indent=2)}")

First chunk:
  Text length: 1877 characters
  Metadata: {
  "file_path": "/Users/apple/Documents/GitHub Repositories/Lexi-Bot/notebooks/../data/txt/A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_name": "A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt",
  "file_type": "text/plain",
  "file_size": 20795,
  "creation_date": "2026-03-12",
  "last_modified_date": "2026-03-12"
}


In [9]:
# Print first 300 characters of first 3 chunks
if nodes:
    for i, node in enumerate(nodes[:3], 1):
        print(f"\nChunk {i}:")
        print(f"  {node.text[:300]}...")


Chunk 1:
  A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025



Author: Vikram Nath Bench: Vikram Nath

	2025 INSC 443	REPORTABLE

IN THE SUPREME COURT OF INDIA CIVIL APPELLATE JU...

Chunk 2:
  Needless to say, that the forests form the lungs of the ecosystem, and any depletion/destruction of forest areas has a direct impact on the entire environment. The world at large is facing the calamities caused by the climate change, and the primary culprit behind this is the depleting forest cover ...

Chunk 3:
  “29. Forests not only provide for and facilitate the sustenance of life, but they also continue to protect and foster it. They continue to tackle the ever-increasing carbon dioxide emissions produced by humans in the name of development, while striving to sustain all species. Despite the unblemished...


### Compare Different Chunk Sizes

In [10]:
# Compare chunk counts for different chunk sizes
if documents:
    chunk_sizes = [256, 512, 1024]
    
    print("Chunk size comparison:")
    print(f"{'Chunk Size':<15} {'Total Chunks':<15} {'Avg Chunks/Doc':<15}")
    print("-" * 45)
    
    for size in chunk_sizes:
        test_splitter = SentenceSplitter(chunk_size=size, chunk_overlap=50)
        test_nodes = test_splitter.get_nodes_from_documents(documents)
        avg_chunks = len(test_nodes) / len(documents)
        print(f"{size:<15} {len(test_nodes):<15} {avg_chunks:<15.1f}")

Chunk size comparison:
Chunk Size      Total Chunks    Avg Chunks/Doc 
---------------------------------------------
256             152             38.0           
512             57              14.2           
1024            26              6.5            


### Chunk Distribution by Source

In [11]:
# Count chunks per source file
if nodes:
    from collections import defaultdict
    
    chunks_by_file = defaultdict(int)
    for node in nodes:
        file_name = node.metadata.get('file_name', 'unknown')
        chunks_by_file[file_name] += 1
    
    print("Chunks per source file:")
    for file_name, count in sorted(chunks_by_file.items()):
        print(f"  {file_name}: {count} chunks")

Chunks per source file:
  A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt: 12 chunks
  A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt: 13 chunks
  A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt: 15 chunks
  A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt: 17 chunks


### Summary Statistics

In [12]:
# Print chunking summary
if nodes and documents:
    total_chars = sum(len(node.text) for node in nodes)
    min_chunk = min(len(node.text) for node in nodes)
    max_chunk = max(len(node.text) for node in nodes)
    avg_chunk = total_chars // len(nodes)
    
    print("=== Chunking Summary ===")
    print(f"\nInput:")
    print(f"  Documents: {len(documents)}")
    print(f"  Total characters: {sum(len(doc.text) for doc in documents):,}")
    print(f"\nOutput:")
    print(f"  Total chunks: {len(nodes)}")
    print(f"  Chunks per doc (avg): {len(nodes) / len(documents):.1f}")
    print(f"\nChunk sizes:")
    print(f"  Min: {min_chunk:,} characters")
    print(f"  Max: {max_chunk:,} characters")
    print(f"  Avg: {avg_chunk:,} characters")

=== Chunking Summary ===

Input:
  Documents: 4
  Total characters: 93,919

Output:
  Total chunks: 57
  Chunks per doc (avg): 14.2

Chunk sizes:
  Min: 322 characters
  Max: 2,165 characters
  Avg: 1,704 characters
